In [7]:
import pandas as pd
import os
import dotenv
import pydoc
import psycopg2
import pyodbc
from dotenv import load_dotenv
from psycopg2.extras import execute_values

load_dotenv()


True

In [8]:
import pyodbc

sql_url = (
    "DRIVER=ODBC Driver 18 for SQL Server;"
    "SERVER=10.1.0.146,1433;"
    "DATABASE=CRMPROD;"
    "UID=readuser;"
    f"PWD={os.getenv('CRM_DB_PASSWORD')};"
    "Trusted_Connection=no;"
    "TrustServerCertificate=yes;"
)

sql_conn = pyodbc.connect(sql_url)

sql_cursor = sql_conn.cursor()

In [9]:
psg_conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="postgres",
    user="postgres",
    password=os.getenv("POSTGRES_PASSWORD")
)

psg_cursor = psg_conn.cursor()

psg_cursor.execute("SELECT VERSION();")

psg_version = psg_cursor.fetchone()[0]

psg_version

'PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit'

In [10]:
CRM_TABLES = {

    # ── Item / product master ─────────────────────────────────────────────
    "item_master": [
        "itemmasters",                      # SKU master (code, description, uom, enabled_flag)
        "ItemCategories",                   # segment1..segment4 (Division/Business/Category/Family)
        "PurchaseRequisitionPtoPts",        # PTO / PTS classification (date-effective)
    ],

    # ── Customer & sales-territory master ─────────────────────────────────
    "customer_master": [
        "CustomerMasters",                  # customer_id / number / name
        "CustomerSites",                    # site -> mc_code (market circle), primary_flag
        "CustomerClassificationHeaders",    # customer classification header
        "CustomerClassificationDetails",    # Class / SubClass / Account_Year
        "MarketCircles",                    # mc_code, region, collector_id
        "Collectors",                       # collector id -> name
    ],

    # ── Sales orders / SOC (open order book) ──────────────────────────────
    "sales_order_soc": [
        "SaleOrderHdrs",                    # SOC header
        "SaleOrderdtls",                    # SOC lines (status = 'OPEN')
        "SocPendingDetails",                # CRM's daily pending-SOC snapshot (Commit Risk)
        # "FnOrderDtlPending",              # TVF - pending order header
        # "FnScheduleDtlPending",           # TVF - pending schedule lines (balance qty)
    ],

    # ── Dispatch / billing history ────────────────────────────────────────
    "dispatch": [
                "DispatchDetails",                  # dispatched qty per SOC line (what shipped)
                "Dispatches",                       # dispatch note header: invoice no/date, status, cancel flag
                "Schedules",                        # planned dispatch per SOC line, with reschedules and status
                "SocCancelDetails",                 # cancelled / closed SOC lines, remaining qty and reason
                "DeliveryFroms",
        # "FnDespatchDetails",              # TVF - dispatch cube (item x customer x collector x MC)
    ],

    # ── Quotation / pipeline ──────────────────────────────────────────────
    "quotation": [
        "QuotationHdrs",                    # quote header (status_id, customer, collector)
        "QuotationDtls",                    # quote lines (qty, value)
        "QuotationStatus",                  # status master (open vs won/lost)
        # "FnQuotationDetails",             # TVF - quote details (legacy adapter path)
    ],

    # ── Business plan / projection (S&OP demand) ──────────────────────────
    "business_plan": [
        "SCBusinessMonthlyPlanHdrs",        # plan header + annual potential/budget + JC status
        "SCBusinessMonthlyPlanDtls",        # per-JC week1/week2 user-defined qty
        "SCBusinessMonthlyPlanJCDtls",      # next-month-1 / next-month-2 JC qty
        "JourneyCalendars",                 # JC master (name, effective_from/to, active/closed)
    ],

    # ── Procurement / purchase ────────────────────────────────────────────
    "purchase": [
        "BiPoDetails",                      # open PO in-transit (ordered - received - cancelled)
        "PurchaseRequisitionHdrs",          # requisition header (supplier)
        "PurchaseRequisitionDtls",          # requisition lines: unit_price vs lastpoprice (RM price)
    ],

    # ── Inventory / stock ─────────────────────────────────────────────────
    "inventory": [
        "BiStockDetail",                    # on-hand by org / subinv / lot + aging + item cost
    ],

    # ── Users, roles & data-scope mappings ────────────────────────────────
    "user_and_scope": [
        "Users",                            # dbo.Users - CRM user master
        "UserRoles",                        # user -> role link
        "Roles",                            # role master (Technical Head, Business Head, ...)
        "UserMarketCircleMappings",         # Sales Executive -> market circle
        "UserCustomerMappings",             # Technical Executive -> customer
        "TechnicalUserSegmentMappings",     # Technical Head/Manager -> segment + collectors
        "CollectorMailMappings",            # Branch Manager / Regional Manager -> collectors
        "SpAlertSegmentWorkflowHdrs",       # Division Head -> segment2
        "SpAlertSegmentWorkflowDtls",       # Business Head -> segment3/4 + collectors
    ],
}



In [11]:
info = {}

for category, tables in CRM_TABLES.items():
    for table in CRM_TABLES.get(category):
        query = f"""
            SELECT COUNT(*)
            FROM [CRMPROD].[dbo].[{table}]
        """

        sql_cursor.execute(query)
        info[table] = sql_cursor.fetchone()[0]

        print(f"{table}  : {info[table]}")
        print("*"*50)

# print(info)

itemmasters  : 26115
**************************************************
ItemCategories  : 26150
**************************************************
PurchaseRequisitionPtoPts  : 180365
**************************************************
CustomerMasters  : 84677
**************************************************
CustomerSites  : 227163
**************************************************
CustomerClassificationHeaders  : 8020
**************************************************
CustomerClassificationDetails  : 8020
**************************************************
MarketCircles  : 254
**************************************************
Collectors  : 129
**************************************************
SaleOrderHdrs  : 1109340
**************************************************
SaleOrderdtls  : 2512981
**************************************************
SocPendingDetails  : 14264
**************************************************
DispatchDetails  : 2491312
***************************************

In [12]:
import pandas as pd

output_file = "CRM_Item_Master_Data.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for table in CRM_TABLES.get("dispatch", []):

        print(f"Processing: {table}")

        sql_query = f"""
            SELECT TOP (100) *
            FROM [CRMPROD].[dbo].[{table}]
        """

        try:
            sql_cursor.execute(sql_query)

            # Get column names
            columns = [col[0] for col in sql_cursor.description]

            # Fetch rows
            data = sql_cursor.fetchall()

            print(f"Rows: {len(data)}")
            print(f"Columns: {len(columns)}")

            # Convert to DataFrame
            df = pd.DataFrame.from_records(data, columns=columns)

            # Excel sheet names have a maximum length of 31 characters
            sheet_name = table[:31]

            df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )

            print(f"✓ {table} written successfully")

        except Exception as e:
            print(f"✗ Error in {table}: {e}")

print(f"\nExcel file created: {output_file}")

Processing: DispatchDetails
Rows: 100
Columns: 48
✓ DispatchDetails written successfully
Processing: Dispatches
Rows: 100
Columns: 61
✓ Dispatches written successfully
Processing: Schedules
Rows: 100
Columns: 63
✓ Schedules written successfully
Processing: SocCancelDetails
Rows: 100
Columns: 26
✓ SocCancelDetails written successfully
Processing: DeliveryFroms
Rows: 46
Columns: 8
✓ DeliveryFroms written successfully

Excel file created: CRM_Item_Master_Data.xlsx
